# Conformal Invariants as Features for Geometric Deep Learning

This notebook demonstrates extracting discrete conformal invariants from triangle meshes
and using them as features for shape classification.

## Background

Conformal geometry studies properties preserved under angle-preserving transformations.
The key insight for ML: conformal invariants capture richer geometric information than
standard isometry invariants (Gaussian/mean curvature), particularly for deformable shapes.

Our feature set includes:
- **Conformal factor** — from discrete Yamabe flow (uniformization)
- **Willmore density** — H² (conformal invariant energy density)
- **Q-curvature** (orders 2 and 4) — higher-order curvature invariants
- **Bach tensor norm** — 4th-order curvature obstruction
- **Cross-ratios** — Möbius-invariant discrete conformal structure (Bobenko-Pinkall-Springborn)

References:
- Blitz, "Conformal Hypersurface Geometry" (PhD thesis, UC Davis)
- Bobenko, Pinkall, Springborn, "Discrete conformal maps and ideal hyperbolic polyhedra" (2015)

In [ ]:
import torch
import math

def make_icosphere(subdivisions=3):
    phi = (1 + math.sqrt(5)) / 2
    verts = [
        [-1, phi, 0], [1, phi, 0], [-1, -phi, 0], [1, -phi, 0],
        [0, -1, phi], [0, 1, phi], [0, -1, -phi], [0, 1, -phi],
        [phi, 0, -1], [phi, 0, 1], [-phi, 0, -1], [-phi, 0, 1],
    ]
    faces = [
        [0,11,5],[0,5,1],[0,1,7],[0,7,10],[0,10,11],
        [1,5,9],[5,11,4],[11,10,2],[10,7,6],[7,1,8],
        [3,9,4],[3,4,2],[3,2,6],[3,6,8],[3,8,9],
        [4,9,5],[2,4,11],[6,2,10],[8,6,7],[9,8,1],
    ]
    vertices = torch.tensor(verts, dtype=torch.float64)
    triangles = torch.tensor(faces, dtype=torch.long)
    vertices = vertices / vertices.norm(dim=1, keepdim=True)
    for _ in range(subdivisions):
        vertices, triangles = subdivide(vertices, triangles)
    return vertices, triangles

def subdivide(vertices, faces):
    edge_map = {}
    new_verts = list(vertices)
    def get_midpoint(i, j):
        key = (min(i, j), max(i, j))
        if key in edge_map:
            return edge_map[key]
        mid = (vertices[i] + vertices[j]) / 2
        mid = mid / mid.norm()
        idx = len(new_verts)
        new_verts.append(mid)
        edge_map[key] = idx
        return idx
    new_faces = []
    for f in faces:
        a, b, c = f[0].item(), f[1].item(), f[2].item()
        ab, bc, ca = get_midpoint(a, b), get_midpoint(b, c), get_midpoint(c, a)
        new_faces.extend([[a, ab, ca], [b, bc, ab], [c, ca, bc], [ab, bc, ca]])
    return torch.stack(new_verts), torch.tensor(new_faces, dtype=torch.long)

# Unit sphere
sphere_v, sphere_f = make_icosphere(3)
print(f"Sphere: {sphere_v.shape[0]} vertices, {sphere_f.shape[0]} faces")

# Ellipsoid (stretched sphere)
ellipsoid_v = sphere_v.clone()
ellipsoid_v[:, 0] *= 2.0  # stretch x-axis
print(f"Ellipsoid: {ellipsoid_v.shape[0]} vertices, {ellipsoid_v.shape[0]} faces")

In [ ]:
from conformal_features.features.pipeline import mesh_conformal_features

# Extract features for both shapes
sphere_feats = mesh_conformal_features(sphere_v, sphere_f, include_isometry=True)
ellipsoid_feats = mesh_conformal_features(ellipsoid_v, sphere_f, include_isometry=True)

feature_names = [
    'conformal_factor', 'willmore_density', 'Q_2', 'Q_4',
    'bach_norm', 'cross_ratio_mean', 'cross_ratio_var',
    'gaussian_K', 'mean_H', 'H\u00b2-K'
]

print(f"Feature shape (sphere): {sphere_feats.shape}")
print(f"Feature shape (ellipsoid): {ellipsoid_feats.shape}")
print(f"\nPer-feature statistics (sphere):")
for i, name in enumerate(feature_names):
    col = sphere_feats[:, i]
    print(f"  {name:20s}: mean={col.mean():.4f}, std={col.std():.4f}, "
          f"min={col.min():.4f}, max={col.max():.4f}")

## Feature Comparison: Sphere vs Ellipsoid

On a perfect sphere, most conformal features are constant (uniform curvature).
On an ellipsoid, features vary — capturing the non-uniform conformal structure.

This is what makes conformal features informative for shape classification:
they detect geometric deformations that isometry-only features might miss.

In [ ]:
print("Feature comparison (mean \u00b1 std):")
print(f"{'Feature':20s} | {'Sphere':25s} | {'Ellipsoid':25s}")
print("-" * 75)
for i, name in enumerate(feature_names):
    s = sphere_feats[:, i]
    e = ellipsoid_feats[:, i]
    print(f"{name:20s} | {s.mean():10.4f} \u00b1 {s.std():.4f}      | "
          f"{e.mean():10.4f} \u00b1 {e.std():.4f}")

In [ ]:
# Verify rotation invariance
c, s = torch.cos(torch.tensor(1.57)), torch.sin(torch.tensor(1.57))
R = torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=sphere_v.dtype)
rotated_v = sphere_v @ R.T

rotated_feats = mesh_conformal_features(rotated_v, sphere_f, include_isometry=False)
original_feats = mesh_conformal_features(sphere_v, sphere_f, include_isometry=False)

max_diff = (rotated_feats - original_feats).abs().max().item()
print(f"Max feature difference under 90\u00b0 rotation: {max_diff:.2e}")
print(f"Rotation invariant: {'YES' if max_diff < 1e-4 else 'NO'}")

## Feature Extraction for GNN Training

The `mesh_conformal_features()` function returns a `(V, D)` tensor that can be used
directly as node features in a GNN (e.g., DiffusionNet).

```python
from conformal_features.features.pipeline import mesh_conformal_features

# For each mesh in your dataset:
features = mesh_conformal_features(vertices, faces, include_isometry=True)
# features.shape = (num_vertices, 10)
# Feed directly into DiffusionNet or any per-vertex GNN
```

### Benchmark Scripts

Run our benchmarks to compare feature sets:

```bash
# ShapeNet classification
python -m conformal_features.benchmarks.shapenet_classify --data_dir /path/to/shapenet --features conformal

# SHREC retrieval
python -m conformal_features.benchmarks.shrec_retrieval --data_dir /path/to/shrec --features conformal

# FAUST correspondence
python -m conformal_features.benchmarks.faust_correspondence --data_dir /path/to/faust --features conformal
```

In [ ]:
import time

verts, faces = make_icosphere(2)  # smaller mesh for timing
times = []
for _ in range(5):
    start = time.time()
    _ = mesh_conformal_features(verts, faces, include_isometry=True)
    times.append(time.time() - start)

print(f"Feature extraction time ({verts.shape[0]} vertices):")
print(f"  Mean: {sum(times)/len(times):.3f}s")
print(f"  Min:  {min(times):.3f}s")
print(f"  Max:  {max(times):.3f}s")